
## Understand the Dataset and ER Diagram


# Step 1 – Overview of the Dataset

In [ ]:

The Cart2 Insight project uses an e-commerce dataset containing 9 related tables. 
These tables capture information about customers, orders, products, sellers, payments, reviews, product categories, and geographical locations.

The main purpose of understanding the dataset and Entity-Relationship (ER) structure is to identify how the tables are connected and 
determine the correct keys and relationships before performing data cleaning, SQL loading, analysis, and feature engineering.

The nine tables are:

1. Customers
2. Geolocation
3. Orders
4. Order Items
5. Order Payments
6. Order Reviews
7. Products
8. Sellers
9. Product Category Translation



# Step 2  Understanding the 9 Tables

In [ ]:
## 1. Customers

The `customers` table contains information about customers who placed orders.

**Important columns:**

| Column                   | Data Type | Description                                           | Key         |
| ------------------------ | --------- | ----------------------------------------------------- | ----------- |
| customer_id              | VARCHAR   | Unique identifier for the customer/order record       | Primary Key |
| customer_unique_id       | VARCHAR   | Identifier associated with the customer across orders | —           |
| customer_zip_code_prefix | INT       | Customer ZIP-code prefix                              | Foreign Key           |
| customer_city            | VARCHAR   | Customer city                                         | —           |
| customer_state           | VARCHAR   | Customer state                                        | —           |

**Rows:** 99,441
**Primary Key:** `customer_id`

---

## 2. Geolocation

The `geolocation` table contains geographical information associated with Brazilian ZIP-code prefixes.

| Column                      | Data Type | Description     | Key |
| --------------------------- | --------- | --------------- | -------------|
| geolocation_zip_code_prefix | INT       | ZIP-code prefix | Primary Key  |
| geolocation_lat             | DECIMAL   | Latitude        | —            |
| geolocation_lng             | DECIMAL   | Longitude       | —            |
| geolocation_city            | VARCHAR   | City            | —            |
| geolocation_state           | VARCHAR   | State           | —            |

**Rows:** 738,327

The ZIP-code prefix is **not unique**, so it was not used as a primary key.
Check	Result
Total rows	1,000,163
Unique zip_code_prefix values	19,015
Prefixes appearing more than once	17,972
Prefixes appearing exactly once	1,043

Only about 5% of ZIP prefixes are unique in this table. The rest repeat dozens or even hundreds of times — 
for example, ZIP prefix 24220 appears 1,146 times, and 24230 appears 1,102 times, each with different latitude/longitude values.

This confirms the original statement: since zip_code_prefix maps to many rows (many distinct lat/lng points, and occasionally 
slightly different city/state entries), it cannot serve as a primary key on its own — a primary key must uniquely identify each row, 
and this column doesn't.
---

## 3. Orders

The `orders` table contains the main order-level information and connects customers with the order lifecycle.

| Column                        | Data Type | Description                                      | Key         |
| ----------------------------- | --------- | ------------------------------------------------ | ----------- |
| order_id                      | VARCHAR   | Unique order identifier                          | Primary Key |
| customer_id                   | VARCHAR   | Customer associated with the order               | Foreign Key |
| order_status                  | VARCHAR   | Current order status                             | —           |
| order_purchase_timestamp      | DATETIME  | Date and time when order was placed              | —           |
| order_approved_at             | DATETIME  | Payment approval timestamp                       | —           |
| order_delivered_carrier_date  | DATETIME  | Date order was handed to carrier                 | —           |
| order_delivered_customer_date | DATETIME  | Date order reached customer                      | —           |
| order_estimated_delivery_date | DATETIME  | Estimated delivery date                          | —           |


**Rows:** 99,441
**Primary Key:** `order_id`
**Foreign Key:** `customer_id → customers.customer_id`

---

## 4. Order Items

The `order_items` table contains individual products included in each order.

| Column              | Data Type | Description                   | Key                 |
| ------------------- | --------- | ----------------------------- | ------------------- |
| order_id            | VARCHAR   | Order identifier              | Foreign Key         |
| order_item_id       | INT       | Item sequence within an order | Primary Key         |
| product_id          | VARCHAR   | Product identifier            | Foreign Key         |
| seller_id           | VARCHAR   | Seller identifier             | Foreign Key         |
| shipping_limit_date | DATETIME  | Seller shipping deadline      | —                   |
| price               | DECIMAL   | Product price                 | —                   |
| freight_value       | DECIMAL   | Freight/shipping cost         | —                   |

**Rows:** 112,650
**Primary Key:** `(order_id, order_item_id)`
**Foreign Keys:**
* `order_id → orders.order_id`
* `product_id → products.product_id`
* `seller_id → sellers.seller_id`

---

## 5. Order Payments

The `order_payments` table contains payment information for orders.

| Column               | Data Type | Description                | Key                 |
| -------------------- | --------- | -------------------------- | ------------------- |
| order_id             | VARCHAR   | Order identifier           | Foreign Key         |
| payment_sequential   | INT       | Sequence number of payment | Primary Key         |
| payment_type         | VARCHAR   | Payment method             | —                   |
| payment_installments | INT       | Number of installments     | —                   |
| payment_value        | DECIMAL   | Payment amount             | —                   |

**Rows:** 103,886
**Primary Key:** `(order_id, payment_sequential)`
**Foreign Key:** `order_id → orders.order_id`

**Important project note:
** `payment_value` is used for payment analysis but **not** for calculating total order revenue.
    For this project, total order value is calculated as:
`   price + freight_value`
    summed across the items belonging to an order.
---

## 6. Order Reviews

The `order_reviews` table contains customer review information.

| Column                  | Data Type    | Description                                         | Key         |
| ----------------------- | ------------ | --------------------------------------------------- | ----------- |
| review_id               | VARCHAR      | Review identifier                                   | Primary Key |          |
| order_id                | VARCHAR      | Associated order                                    | Foreign Key |
| review_score            | INT          | Customer rating from 1 to 5                         | —           |
| review_comment_title    | VARCHAR/TEXT | Review title                                        | —           |
| review_comment_message  | TEXT         | Review message                                      | —           |
| review_creation_date    | DATETIME     | Review creation date                                | —           |
| review_answer_timestamp | DATETIME     | Review response timestamp                           | —           |

**Rows:** 99,224
**Primary Key:** review_id  
**Foreign Key:** order_id → orders.order_id
---
## 7. Products

The `products` table contains information about products sold on the platform.

| Column                        | Data Type | Description                   | Key         |
| ----------------------------- | --------- | ----------------------------- | ----------- |
| product_id                    | VARCHAR   | Unique product identifier     | Primary Key |
| product_category_name         | VARCHAR   | Original product category     | Foreign Key |
| product_name_length           | INT       | Length of product name        | —           |
| product_description_length    | INT       | Length of product description | —           |
| product_photos_qty            | INT       | Number of product photographs | —           |
| product_weight_g              | DECIMAL   | Product weight in grams       | —           |
| product_length_cm             | DECIMAL   | Product length                | —           |
| product_height_cm             | DECIMAL   | Product height                | —           |
| product_width_cm              | DECIMAL   | Product width                 | —           |


**Rows:** 32,951

**Primary Key:** product_id
**Foreign Key:** product_category_name 
product_category_name → product_category_translation.product_category_name

---

## 8. Sellers

The `sellers` table contains information about sellers.

| Column                 | Data Type | Description              | Key         |
| ---------------------- | --------- | ------------------------ | ----------- |
| seller_id              | VARCHAR   | Unique seller identifier | Primary Key |
| seller_zip_code_prefix | INT       | Seller ZIP-code prefix   | Foreign Key |
| seller_city            | VARCHAR   | Seller city              | —           |
| seller_state           | VARCHAR   | Seller state             | —           |

**Rows:** 3,095

**Primary Key:** seller_id
**Foreign Key:** seller_zip_code_prefix
---

## 9. Product Category Translation

The `product_category_translation` table maps the original Portuguese product category names to English category names.

| Column                        | Data Type | Description            | Key         |
| ----------------------------- | --------- | ---------------------- | ----------- |
| product_category_name         | VARCHAR   | Original category name | Primary Key |
| product_category_name_english | VARCHAR   | English category name  | —           |

**Rows:** 73
**Primary Key:** `product_category_name`

Two categories referenced by products were added during SQL preparation because they were missing 
from the original cleaned category file. This allowed the product-category foreign-key relationship to be maintained.

---

#  Step 3 Identify Primary Keys and  Foreign Keys

In [ ]:
# Identify Primary Keys

Primary keys uniquely identify records within each table.

| Table                        | Primary Key                      |
| ---------------------------- | -------------------------------- |
| customers                    |  customer_id                     |
| geolocation                  |  geolocation_zip_code_prefix     |
| orders                       |  order_id                        |
| order_items                  |  order_item_id                   |
| order_payments               |  payment_sequential              |
| order_reviews                |  review_id                       |
| products                     |  product_id                      |
| sellers                      |  seller_id                       |
| product_category_translation |  product_category_name           |

---

# Identify Foreign Keys

The major foreign-key relationships in the database are:

| Child Table    | Foreign Key                | Parent Table                 | Parent Key              |
| -------------- | -----------------------    | ---------------------------- | ----------------------- |
| orders         | customer_zip_code_prefix   | customers                    | customer_id             |
| order_items    | order_id                   | orders                       | order_id                |
| order_items    | product_id                 | products                     | product_id              |
| order_items    | seller_id                  | sellers                      | seller_id               |
| order_payments | order_id                   | orders                       | order_id                |
| order_reviews  | order_id                   | orders                       | order_id                | 
| products       | product_category_name      | category_translation         | product_category_name   |
| sellers        | seller_zip_code_prefix     | geolocation                  | geolocation_zip_code_prefix|
| customers      | customer_zip_code_prefix   | geolocation                  | geolocation_zip_code_prefix|
    

All relationship in the table above is a valid, enforceable foreign key, since each parent key listed is a true primary key.

---

#  Step 4 Understanding the Relationships between tables

In [ ]:

Understanding how these tables relate to one another is essential before any cleaning, warehousing, or analysis can begin.

The nine tables in this dataset are not independent — together, they form a single, connected structure that traces 
the complete lifecycle of an order, from the moment a customer places it to the moment they leave a review.


1. Customers Dataset - olist_customers_dataset.csv
   Key Field: customer_id
   Links to:  Orders Dataset via customer_id → identifies which orders belong to which customer.
   Insight:  Helps track churn, repeat purchases, and customer segmentation.

2. Orders Dataset - olist_orders_dataset.csv
   Key Field:   order_id
   Links to:   
                Customers Dataset (customer_id)
                Order Items Dataset (order_id)
                Order Payments Dataset (order_id)
                Order Reviews Dataset (order_id)
   Insight:     Central hub table — connects customers, items, payments, and reviews.

3. Order Items Dataset - olist_order_items_dataset.csv
   Key Fields:  order_id, product_id, seller_id
   Links to:
                Orders Dataset (order_id)
                Products Dataset (product_id)
                Sellers Dataset (seller_id)

    Insight:    Defines what products were bought, from which seller, in each order.

4. Products Dataset - olist_products_dataset.csv
   Key Field:   product_id
   Links to:
                Order Items Dataset (product_id)
                Product Category Translation Dataset (product_category_name)
   Insight:     Provides product details and category mapping for analysis.

5. Product Category Translation - product_category_name_translation.csv
   Key Field:   product_category_name
   Links to:
                Products Dataset (product_category_name)
   Insight:     Translates Portuguese product categories into English for easier reporting.

6. Order Payments Dataset - olist_order_payments_dataset.csv)\
   Key Field:   order_id
   Links to:
                Orders Dataset (order_id)
   Insight:     Tracks payment methods, installments, and amounts.

7. Order Reviews Dataset -olist_order_reviews_dataset.csv
   Key Field:   order_id
   Links to:
                Orders Dataset (order_id)
   Insight:     Captures customer feedback, ratings, and review timestamps.

8. Sellers Dataset - olist_sellers_dataset.csv
   Key Field:   seller_id
   Links to:
   Order Items Dataset (seller_id)
   Insight:    Provides seller location and identity, useful for seller performance analysis.

9. Geolocation Dataset - olist_geolocation_dataset.csv
   Key Fields: geolocation_zip_code_prefix
   Links to:
               Customers Dataset (customer_zip_code_prefix)
               Sellers Dataset (seller_zip_code_prefix)
   Insight:    Enables mapping of customer and seller locations based on ZIP-code prefixes for delivery optimization.
---


##  Step 5 ER Diagram

In [ ]:
## ER Diagram
[ER Diagram](https://shadiyapp.github.io/Decoding-E-Commerce-Performance/)


In [ ]:
                         ┌─────────────────────────────┐
                         │          CUSTOMERS          │
                         ├─────────────────────────────┤
                         │ PK customer_id              │
                         │ customer_unique_id          │
                         │ FK customer_zip_code_prefix │
                         │ customer_city               │
                         │ customer_state              │
                         └──────────────┬──────────────┘
                                        │
                                        │ 1 : N
                                        ▼
                         ┌─────────────────────────────┐
                         │           ORDERS            │
                         ├─────────────────────────────┤
                         │ PK order_id                 │
                         │ FK customer_id              │
                         │ order_status                │
                         │ order_purchase_timestamp    │
                         │ order_approved_at           │
                         │ order_delivered_carrier_date│
                         │ order_delivered_customer_date│
                         │ order_estimated_delivery_date│
                         │ actual_delivery_days       │
                         │ estimated_delivery_days    │
                         │ delivery_delay_days        │
                         └──────┬─────────┬───────────┘
                                │         │
                   1 : N        │         │ 1 : N
                                │         │
                ┌───────────────┘         └────────────────┐
                ▼                                          ▼
┌──────────────────────────────┐          ┌─────────────────────────────┐
│        ORDER_ITEMS           │          │       ORDER_PAYMENTS        │
├──────────────────────────────┤          ├─────────────────────────────┤
│ FK order_id                  │          │ FK order_id                 │
│ PK order_item_id             │          │ PK payment_sequential       │
│ FK product_id                │          │                             │
│ FK seller_id                 │          │ payment_type                │
│ shipping_limit_date          │          │ payment_installments        │
│ price                        │          │ payment_value               │
│ freight_value                │          └─────────────────────────────┘
└───────┬───────────┬───────── ┘
        │           │
     N : 1        N : 1
        │           │
        ▼           ▼
┌───────────────┐  ┌──────────────────────┐
│   PRODUCTS    │  │       SELLERS        │
├───────────────┤  ├──────────────────────┤
│ PK product_id │  │ PK seller_id         │
│ FK product_   │  │ FK seller_zip_code   │
│ category_     │  │ seller_city          │
│ name          │  │ seller_state         │
│               │  └──────────────────────┘
│ photos_qty    │
│ weight_g      │
│ length_cm     │
│ height_cm     │
│ width_cm      │
└───────┬───────┘
        │
        │ N : 1
        ▼
┌──────────────────────────────────┐
│ PRODUCT_CATEGORY_TRANSLATION     │
├──────────────────────────────────┤
│ PK product_category_name         │
│ product_category_name_english    │
└──────────────────────────────────┘


                         ┌─────────────────────────────┐
                         │       ORDER_REVIEWS         │
                         ├─────────────────────────────┤
                         │ PK review_id                │
                         │ FK order_id                 │
                         │ review_id                   │
                         │ review_score                │
                         │ review_comment_title        │
                         │ review_comment_message      │
                         │ review_creation_date        │
                         │ review_answer_timestamp     │
                         └──────────────▲──────────────┘
                                        │
                                        │ N : 1
                                        │
                                      ORDERS


              ┌─────────────────────────────────────────┐
              │              GEOLOCATION                │
              ├─────────────────────────────────────────┤
              │ geolocation_zip_code_prefix   PK        │
              │ geolocation_lat                         │
              │ geolocation_lng                         │
              │ geolocation_city                        │
              │ geolocation_state                       │
              │                                         │
              │                                         │
              └─────────────────────────────────────────┘

# Step 6 Prepare a Data Dictionary

In [ ]:

A data dictionary provides a structured description of the fields used in the project.

| Table                        | Column                        | Data Type    | Description                                      |
| ---------------------------- | ----------------------------- | ------------ | ------------------------------------------------ |
| customers                    | customer_id                   | VARCHAR      | Unique customer/order record identifier          |
| customers                    | customer_unique_id            | VARCHAR      | Customer-level identifier                        |
| customers                    | customer_zip_code_prefix      | INT          | Customer ZIP-code prefix                         |
| customers                    | customer_city                 | VARCHAR      | Customer city                                    |
| customers                    | customer_state                | VARCHAR      | Customer state                                   |
| geolocation                  | geolocation_zip_code_prefix   | INT          | ZIP-code prefix                                  |
| geolocation                  | geolocation_lat               | DECIMAL      | Geographic latitude                              |
| geolocation                  | geolocation_lng               | DECIMAL      | Geographic longitude                             |
| geolocation                  | geolocation_city              | VARCHAR      | Geographic city                                  |
| geolocation                  | geolocation_state             | VARCHAR      | Geographic state                                 |
| orders                       | order_id                      | VARCHAR      | Unique order identifier                          |
| orders                       | customer_id                   | VARCHAR      | Customer associated with order                   |
| orders                       | order_status                  | VARCHAR      | Current order status                             |
| orders                       | order_purchase_timestamp      | DATETIME     | Order purchase date and time                     |
| orders                       | order_approved_at             | DATETIME     | Payment approval date and time                   |
| orders                       | order_delivered_carrier_date  | DATETIME     | Date handed to carrier                           |
| orders                       | order_delivered_customer_date | DATETIME     | Date delivered to customer                       |
| orders                       | order_estimated_delivery_date | DATETIME     | Estimated delivery date                          |
| orders                       | actual_delivery_days          | INT          | Actual delivery duration                         |
| orders                       | estimated_delivery_days       | INT          | Estimated delivery duration                      |
| orders                       | delivery_delay_days           | INT          | Difference between actual and estimated delivery |
| order_items                  | order_id                      | VARCHAR      | Associated order                                 |
| order_items                  | order_item_id                 | INT          | Item sequence within order                       |
| order_items                  | product_id                    | VARCHAR      | Associated product                               |
| order_items                  | seller_id                     | VARCHAR      | Associated seller                                |
| order_items                  | shipping_limit_date           | DATETIME     | Shipping deadline                                |
| order_items                  | price                         | DECIMAL      | Product price                                    |
| order_items                  | freight_value                 | DECIMAL      | Shipping/freight cost                            |
| order_payments               | order_id                      | VARCHAR      | Associated order                                 |
| order_payments               | payment_sequential            | INT          | Payment sequence                                 |
| order_payments               | payment_type                  | VARCHAR      | Payment method                                   |
| order_payments               | payment_installments          | INT          | Number of installments                           |
| order_payments               | payment_value                 | DECIMAL      | Payment amount                                   |
| order_reviews                | review_pk                     | INT          | Unique review record identifier                  |
| order_reviews                | review_id                     | VARCHAR      | Review identifier                                |
| order_reviews                | order_id                      | VARCHAR      | Associated order                                 |
| order_reviews                | review_score                  | INT          | Customer rating from 1 to 5                      |
| order_reviews                | review_comment_title          | VARCHAR/TEXT | Review title                                     |
| order_reviews                | review_comment_message        | TEXT         | Review message                                   |
| order_reviews                | review_creation_date          | DATETIME     | Review creation date                             |
| order_reviews                | review_answer_timestamp       | DATETIME     | Review response date                             |
| products                     | product_id                    | VARCHAR      | Unique product identifier                        |
| products                     | product_category_name         | VARCHAR      | Original product category                        |
| products                     | product_category_name_english | VARCHAR      | English product category                         |
| products                     | product_name_length           | INT          | Product-name length                              |
| products                     | product_description_length    | INT          | Product-description length                       |
| products                     | product_photos_qty            | INT          | Number of product photos                         |
| products                     | product_weight_g              | DECIMAL      | Product weight                                   |
| products                     | product_length_cm             | DECIMAL      | Product length                                   |
| products                     | product_height_cm             | DECIMAL      | Product height                                   |
| products                     | product_width_cm              | DECIMAL      | Product width                                    |
| sellers                      | seller_id                     | VARCHAR      | Unique seller identifier                         |
| sellers                      | seller_zip_code_prefix        | INT          | Seller ZIP-code prefix                           |
| sellers                      | seller_city                   | VARCHAR      | Seller city                                      |
| sellers                      | seller_state                  | VARCHAR      | Seller state                                     |
| product_category_translation | product_category_name         | VARCHAR      | Original category name                           |
| product_category_translation | product_category_name_english | VARCHAR      | English category name                            |

---

## Step 7 Understand columns and data types

In [10]:
import pandas as pd
import numpy as np

In [11]:
import os

print(os.listdir())

['.ipynb_checkpoints', '01_data_understanding.ipynb', 'Cart Main2', 'Cart main3', 'Cart main4', 'Cart2Insights Dataset', 'Cart2Insights _ Decoding E-Commerce Performance (1).docx', 'Cart2Insights-main', 'Cart2Insights-main1.zip', 'Cart2main', 'Step 1 Understanding Data.ipynb', 'step1.docx', 'venv', '~$step1.docx']


In [1]:
## Load the data
import pandas as pd

customers = pd.read_csv(r"D:\HCL_PROJECTS\Cart2 Insight\Data\raw\olist_customers_dataset.csv")
geolocation = pd.read_csv(r"D:\HCL_PROJECTS\Cart2 Insight\Data\raw\olist_geolocation_dataset.csv")
order_items = pd.read_csv(r"D:\HCL_PROJECTS\Cart2 Insight\Data\raw\olist_order_items_dataset.csv")
payments = pd.read_csv(r"D:\HCL_PROJECTS\Cart2 Insight\Data\raw\olist_order_payments_dataset.csv")
reviews = pd.read_csv(r"D:\HCL_PROJECTS\Cart2 Insight\Data\raw\olist_order_reviews_dataset.csv")
orders = pd.read_csv(r"D:\HCL_PROJECTS\Cart2 Insight\Data\raw\olist_orders_dataset.csv")
products = pd.read_csv(r"D:\HCL_PROJECTS\Cart2 Insight\Data\raw\olist_products_dataset.csv")
sellers = pd.read_csv(r"D:\HCL_PROJECTS\Cart2 Insight\Data\raw\olist_sellers_dataset.csv")
category_translation = pd.read_csv(r"D:\HCL_PROJECTS\Cart2 Insight\Data\raw\product_category_name_translation.csv")


In [2]:
print("All 9 datasets loaded successfully!")

All 9 datasets loaded successfully!


In [ ]:
## Dataset Overview

Table	                           LOD (Level Of Details)                                                   Purpose

customers	                       Customer record (one per customer ID)                    Stores customer identity and location (ZIP prefix, city,                                                                                                state), and links each customer to their orders

orders	                           Order	                                                Tracks the order lifecycle: status, purchase, approval,                                                                                                 carrier handoff, and delivery timestamps. This is the core                                                                                              table for delivery analysis

order_items	                       Item within an order                                     Records each product purchased, with the seller, price, and freight cost.                                                                               One order can contain several items and sellers



order_payments	                   Payment entry within an order                    	    Captures payment type, installments, and amount paid. An                                                                                                order can be split across multiple payment methods


order_reviews	                    Review	                                                Holds the customer's review score, comment, and response                                                                                                timestamps, which measure satisfaction


products	                        Product	                                                Describes each product's category and physical attributes                                                                                               (weight, dimensions)


sellers	                            Seller	                                                Identifies each seller and their location, for seller-level                                                                                             revenue and performance analysis


geolocation	                        Geolocation record	                                    Maps ZIP prefixes to latitude and longitude, for location-                                                                                              based delivery and sales analysis. It may contain multiple                                                                                              rows per ZIP prefix


product_category_name_translation	Category	                                            Translates raw category names into readable English names                                                                                               for cleaner charts and reporting

In [3]:
## Data Sets
datasets = [customers, geolocation, order_items, payments, reviews, orders, products, sellers, category_translation]

for i, df in enumerate(datasets, 1):
    print(f"Dataset {i}: Rows = {df.shape[0]}, Columns = {df.shape[1]}")

Dataset 1: Rows = 99441, Columns = 5
Dataset 2: Rows = 1000163, Columns = 5
Dataset 3: Rows = 112650, Columns = 7
Dataset 4: Rows = 103886, Columns = 5
Dataset 5: Rows = 99224, Columns = 7
Dataset 6: Rows = 99441, Columns = 8
Dataset 7: Rows = 32951, Columns = 9
Dataset 8: Rows = 3095, Columns = 4
Dataset 9: Rows = 71, Columns = 2


In [4]:
import pandas as pd

dataset_names = [
    "customers",
    "geolocation",
    "order_items",
    "payments",
    "reviews",
    "orders",
    "products",
    "sellers",
    "category_translation"
]

dataset_list = [
    customers,
    geolocation,
    order_items,
    payments,
    reviews,
    orders,
    products,
    sellers,
    category_translation
]

dataset_overview = pd.DataFrame({
    "table": dataset_names,
    "rows": [df.shape[0] for df in dataset_list],
    "columns": [df.shape[1] for df in dataset_list]
})

dataset_overview

,table,rows,columns
0,customers,99441,5
1,geolocation,1000163,5
2,order_items,112650,7
3,payments,103886,5
4,reviews,99224,7
5,orders,99441,8
6,products,32951,9
7,sellers,3095,4
8,category_translation,71,2


In [5]:
## Data Type
for i, df in enumerate(datasets, 1):
    print(f"\nDataset {i}")
    for column in df.columns:
        print(column, ":", df[column].dtype)


Dataset 1
customer_id : str
customer_unique_id : str
customer_zip_code_prefix : int64
customer_city : str
customer_state : str

Dataset 2
geolocation_zip_code_prefix : int64
geolocation_lat : float64
geolocation_lng : float64
geolocation_city : str
geolocation_state : str

Dataset 3
order_id : str
order_item_id : int64
product_id : str
seller_id : str
shipping_limit_date : str
price : float64
freight_value : float64

Dataset 4
order_id : str
payment_sequential : int64
payment_type : str
payment_installments : int64
payment_value : float64

Dataset 5
review_id : str
order_id : str
review_score : int64
review_comment_title : str
review_comment_message : str
review_creation_date : str
review_answer_timestamp : str

Dataset 6
order_id : str
customer_id : str
order_status : str
order_purchase_timestamp : str
order_approved_at : str
order_delivered_carrier_date : str
order_delivered_customer_date : str
order_estimated_delivery_date : str

Dataset 7
product_id : str
product_category_name : s